In [ ]:
# Document Q&A with Citations using RAG

## Potens AI/ML Internship Assignment

### Author
Samiksha Chougule

---

This notebook implements a Retrieval-Augmented Generation (RAG) system over HR policy documents using Google Gemini and ChromaDB.

Features:
- Document ingestion
- Chunking
- Vector database
- Question Answering
- Citations
- Contradiction Detection
- Multilingual Support

In [ ]:
# 1. Install Dependencies

In [1]:
!pip -q install \
langchain \
langchain-community \
langchain-google-genai \
chromadb \
pypdf \
sentence-transformers \
streamlit \
python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 

In [ ]:
# 2. Import Libraries

In [2]:
import chromadb
import pypdf
import langchain
import google.generativeai

print("Everything imported successfully!")

Everything imported successfully!


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# 3. Upload HR Policy Documents

In [3]:
from google.colab import files

uploaded = files.upload()

Saving Attendance_Policy-2.pdf to Attendance_Policy-2.pdf
Saving Code_of_Conduct-2.pdf to Code_of_Conduct-2.pdf
Saving Employee_Handbook-2.pdf to Employee_Handbook-2.pdf
Saving Leave_Policy-2.pdf to Leave_Policy-2.pdf
Saving WFH_Policy-2.pdf to WFH_Policy-2.pdf


In [ ]:
# 4. Configure Gemini API

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

response = llm.invoke("Say Hello")

print(response.content)

Hello!


In [ ]:
# 5. Load PDF Documents

In [6]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader(".")

documents = loader.load()

print(f"Total pages loaded: {len(documents)}")

# Preview the first page
print("\nDocument:", documents[0].metadata["source"])
print("\nContent Preview:\n")
print(documents[0].page_content[:500])

/tmp/ipykernel_1112/4178440591.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Total pages loaded: 10

Document: Attendance_Policy-2.pdf

Content Preview:

Attendance Policy
Purpose
Attendance records help ensure operational continuity. This policy applies to all full-time employees
unless an exception is approved in writing by Human Resources. Managers are responsible for
communicating these guidelines and employees are expected to comply with them. Failure to follow
the policy may result in corrective action depending on the circumstances. These guidelines are
reviewed periodically to ensure operational consistency and regulatory compliance. This


In [ ]:
# 6. Split Documents into Chunks

In [8]:
import langchain
print(langchain.__version__)

1.3.11


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

print("\nFirst Chunk:\n")
print(chunks[0].page_content)

print("\nMetadata:")
print(chunks[0].metadata)

Total Chunks: 76

First Chunk:

Attendance Policy
Purpose
Attendance records help ensure operational continuity. This policy applies to all full-time employees
unless an exception is approved in writing by Human Resources. Managers are responsible for
communicating these guidelines and employees are expected to comply with them. Failure to follow
the policy may result in corrective action depending on the circumstances. These guidelines are

Metadata:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-07T07:14:10+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-07T07:14:10+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Attendance_Policy-2.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [ ]:
# 7. Generate Embeddings

In [10]:
import chromadb
print(chromadb.__version__)

1.5.9


In [11]:
import google.generativeai as genai
print("Gemini SDK imported successfully")

Gemini SDK imported successfully


In [12]:
import google.generativeai as genai

print(genai.__version__)

0.8.6


In [13]:
print(dir(genai))

['ChatSession', 'GenerationConfig', 'GenerativeModel', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'annotations', 'caching', 'configure', 'create_tuned_model', 'delete_file', 'delete_tuned_model', 'embed_content', 'embed_content_async', 'get_base_model', 'get_file', 'get_model', 'get_operation', 'get_tuned_model', 'list_files', 'list_models', 'list_operations', 'list_tuned_models', 'protos', 'responder', 'string_utils', 'textwrap', 'types', 'update_tuned_model', 'upload_file', 'utils', 'warnings']


In [14]:
import os
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("✅ Gemini configured successfully")

✅ Gemini configured successfully


In [16]:
import google.generativeai as genai

for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(model.name)
        print(model.supported_generation_methods)
        print("-" * 50)

models/gemini-embedding-001
['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']
--------------------------------------------------
models/gemini-embedding-2-preview
['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']
--------------------------------------------------
models/gemini-embedding-2
['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']
--------------------------------------------------


In [17]:
embedding = genai.embed_content(
    model="models/gemini-embedding-001",
    content="Employees receive 12 casual leave days."
)

print(type(embedding))
print(embedding.keys())
print(len(embedding["embedding"]))

<class 'dict'>
dict_keys(['embedding'])
3072


In [ ]:
# 8. Create ChromaDB Vector Store

In [18]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="company_policies"
)

print("✅ Chroma Collection Created")

✅ Chroma Collection Created


In [19]:
documents_list = []
embeddings_list = []
ids = []
metadatas = []

for i, chunk in enumerate(chunks):

    text = chunk.page_content

    embedding = genai.embed_content(
        model="models/gemini-embedding-001",
        content=text
    )["embedding"]

    documents_list.append(text)

    embeddings_list.append(embedding)

    ids.append(f"chunk_{i}")

    metadatas.append({
        "source": chunk.metadata["source"],
        "page": chunk.metadata["page"]
    })

print("Prepared", len(documents_list), "chunks")

Prepared 76 chunks


In [20]:
collection.add(
    documents=documents_list,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas
)

print("✅ Stored", collection.count(), "chunks")

✅ Stored 76 chunks


In [21]:
query = "How many casual leaves are allowed?"

query_embedding = genai.embed_content(
    model="models/gemini-embedding-001",
    content=query
)["embedding"]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(results)

{'ids': [['chunk_50', 'chunk_49', 'chunk_65']], 'embeddings': None, 'documents': [['Employees receive 12 casual leave days each year. This policy supersedes earlier departmental\nguidance. This policy applies to all full-time employees unless an exception is approved in writing\nby Human Resources. Managers are responsible for communicating these guidelines and\nemployees are expected to comply with them. Failure to follow the policy may result in corrective\naction depending on the circumstances. These guidelines are reviewed periodically to ensure', 'Resources. Managers are responsible for communicating these guidelines and employees are\nexpected to comply with them. Failure to follow the policy may result in corrective action depending\non the circumstances. These guidelines are reviewed periodically to ensure operational consistency\nand regulatory compliance.\nCasual Leave\nEmployees receive 12 casual leave days each year. This policy supersedes earlier departmental', 'on the cir

In [ ]:
# 9. Retrieve Relevant Chunks

In [22]:
def retrieve_chunks(question, top_k=3):
    """
    Retrieve the most relevant chunks from ChromaDB.
    """

    query_embedding = genai.embed_content(
        model="models/gemini-embedding-001",
        content=question
    )["embedding"]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results

In [23]:
results = retrieve_chunks(
    "How many casual leaves are allowed?"
)

print(results["documents"][0][0])

Employees receive 12 casual leave days each year. This policy supersedes earlier departmental
guidance. This policy applies to all full-time employees unless an exception is approved in writing
by Human Resources. Managers are responsible for communicating these guidelines and
employees are expected to comply with them. Failure to follow the policy may result in corrective
action depending on the circumstances. These guidelines are reviewed periodically to ensure


In [ ]:
# 10. Build Prompt

In [46]:
def build_prompt(question, retrieved_results):

    context = ""

    docs = retrieved_results["documents"][0]
    metadata = retrieved_results["metadatas"][0]

    for i, (doc, meta) in enumerate(zip(docs, metadata), start=1):

        context += f"""
Document {i}

Source:
{meta['source']}

Page:
{meta['page']+1}

Content:
{doc}

----------------------------------------
"""

    prompt = f"""
You are an intelligent HR Policy Assistant.

You MUST answer ONLY using the provided context.

IMPORTANT RULES

1. Answer ONLY from the retrieved documents.

2. If the answer does not exist, reply EXACTLY:

"I could not find the answer in the provided documents."

3. If multiple documents disagree,
clearly explain the contradiction.

4. Mention the document names whenever referring to evidence.

5. Return the answer in the SAME LANGUAGE as the user's question.

6. Never use outside knowledge.

Question:
{question}

Context:

{context}
"""

    return prompt

In [47]:
prompt = build_prompt(
    "How many casual leaves are allowed?",
    results
)

print(prompt[:2000])


You are an intelligent HR Policy Assistant.

You MUST answer ONLY using the provided context.

IMPORTANT RULES

1. Answer ONLY from the retrieved documents.

2. If the answer does not exist, reply EXACTLY:

"I could not find the answer in the provided documents."

3. If multiple documents disagree,
clearly explain the contradiction.

4. Mention the document names whenever referring to evidence.

5. Return the answer in the SAME LANGUAGE as the user's question.

6. Never use outside knowledge.

Question:
How many casual leaves are allowed?

Context:


Document 1

Source:
Leave_Policy-2.pdf

Page:
1

Content:
Employees receive 12 casual leave days each year. This policy supersedes earlier departmental
guidance. This policy applies to all full-time employees unless an exception is approved in writing
by Human Resources. Managers are responsible for communicating these guidelines and
employees are expected to comply with them. Failure to follow the policy may result in corrective
action d

In [48]:
model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(prompt)

print(response.text)

Documents provide conflicting information regarding the number of casual leaves allowed.

*   Document 1 (Leave_Policy-2.pdf, Page 1) states that "Employees receive 12 casual leave days each year."
*   Document 2 (Leave_Policy-2.pdf, Page 1) also states that "Employees receive 12 casual leave days each year."
*   However, Document 3 (Employee_Handbook-2.pdf, Page 1) states that "Employees are eligible for 10 casual leave days..."


In [49]:
def format_citations(results):
    citations = []

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    seen = set()

    for doc, meta in zip(docs, metas):

        key = (meta["source"], meta["page"])

        if key in seen:
            continue

        seen.add(key)

        citations.append({
            "source": meta["source"],
            "page": meta["page"] + 1,
            "snippet": doc[:200] + "..."
        })

    return citations

In [50]:
citations = format_citations(results)

for citation in citations:
    print(citation)
    print()

{'source': 'Leave_Policy-2.pdf', 'page': 1, 'snippet': 'Employees receive 12 casual leave days each year. This policy supersedes earlier departmental\nguidance. This policy applies to all full-time employees unless an exception is approved in writing\nby Hum...'}

{'source': 'Employee_Handbook-2.pdf', 'page': 1, 'snippet': 'on the circumstances. These guidelines are reviewed periodically to ensure operational consistency\nand regulatory compliance.\nLeave\nEmployees are eligible for 10 casual leave days, 18 annual leave day...'}



In [ ]:
# 11. Question Answering (/ask)

In [51]:
def ask_question(question):
    """
    Complete RAG pipeline with confidence and error handling.
    """

    try:
        # Step 1: Retrieve relevant chunks
        results = retrieve_chunks(question)

        # Step 2: Build prompt
        prompt = build_prompt(question, results)

        # Step 3: Gemini
        model = genai.GenerativeModel("gemini-2.5-flash")

        response = model.generate_content(prompt)

        # Step 4: Citations
        citations = format_citations(results)

        # Step 5: Confidence
        distances = results["distances"][0]

        avg_distance = sum(distances) / len(distances)

        confidence = round(max(0, 1 - avg_distance), 2)

        return {
            "question": question,
            "answer": response.text,
            "confidence": confidence,
            "citations": citations
        }

    except Exception as e:

        return {
            "question": question,
            "error": str(e)
        }

In [52]:
response = ask_question(
    "How many casual leaves are allowed?"
)

response

{'question': 'How many casual leaves are allowed?',
 'answer': 'According to Document 1 (Leave_Policy-2.pdf) and Document 2 (Leave_Policy-2.pdf), employees receive 12 casual leave days each year.\n\nHowever, Document 3 (Employee_Handbook-2.pdf) states that employees are eligible for 10 casual leave days per calendar year.\n\nThere is a contradiction between these documents regarding the number of casual leave days allowed. Documents 1 and 2 indicate 12 days, while Document 3 indicates 10 days.',
 'confidence': 0.44,
 'citations': [{'source': 'Leave_Policy-2.pdf',
   'page': 1,
   'snippet': 'Employees receive 12 casual leave days each year. This policy supersedes earlier departmental\nguidance. This policy applies to all full-time employees unless an exception is approved in writing\nby Hum...'},
  {'source': 'Employee_Handbook-2.pdf',
   'page': 1,
   'snippet': 'on the circumstances. These guidelines are reviewed periodically to ensure operational consistency\nand regulatory complian

In [53]:
def get_document_chunks(document_name):
    """
    Returns all chunks belonging to a specific document.
    """

    results = collection.get(
        where={"source": document_name},
        include=["documents", "metadatas"]
    )

    return results

In [54]:
doc = get_document_chunks("Leave_Policy-2.pdf")

print(len(doc["documents"]))
print(doc["documents"][0][:300])

15
Leave Policy
Purpose
This document defines leave eligibility and approval procedures. This policy applies to all full-time
employees unless an exception is approved in writing by Human Resources. Managers are
responsible for communicating these guidelines and employees are expected to comply with th


In [ ]:
# 12. Document Contradiction Detection (/contradict)

In [55]:
def build_contradiction_prompt(doc1_name, doc2_name):

    doc1 = get_document_chunks(doc1_name)
    doc2 = get_document_chunks(doc2_name)

    text1 = "\n".join(doc1["documents"])
    text2 = "\n".join(doc2["documents"])

    prompt = f"""
You are a document comparison assistant.

Compare the following two HR policy documents.

Document A:
{doc1_name}

{text1}

----------------------------------------

Document B:
{doc2_name}

{text2}

----------------------------------------

Tasks:

1. Determine whether the documents contradict each other.

2. If YES,
list every contradiction.

3. Explain why.

4. Quote the relevant evidence.

5. If there is no contradiction,
say:
"No contradiction found."

Return your answer in clear bullet points.
"""

    return prompt

In [56]:
prompt = build_contradiction_prompt(
    "Employee_Handbook-2.pdf",
    "Leave_Policy-2.pdf"
)

print(prompt[:1500])


You are a document comparison assistant.

Compare the following two HR policy documents.

Document A:
Employee_Handbook-2.pdf

Employee Handbook
Purpose
This handbook explains the company's general employment expectations, culture, and day-to-day
policies. Every employee is expected to read and follow this handbook. This policy applies to all
full-time employees unless an exception is approved in writing by Human Resources. Managers are
responsible for communicating these guidelines and employees are expected to comply with them.
responsible for communicating these guidelines and employees are expected to comply with them.
Failure to follow the policy may result in corrective action depending on the circumstances. These
guidelines are reviewed periodically to ensure operational consistency and regulatory compliance.
This policy applies to all full-time employees unless an exception is approved in writing by Human
Resources. Managers are responsible for communicating these guidelines a

In [57]:
def contradict(doc1, doc2):

    prompt = build_contradiction_prompt(doc1, doc2)

    model = genai.GenerativeModel("gemini-2.5-flash")

    response = model.generate_content(prompt)

    return response.text

In [58]:
print(
    contradict(
        "Employee_Handbook-2.pdf",
        "Leave_Policy-2.pdf"
    )
)

Here's a comparison of the two documents:

*   **YES, the documents contradict each other.**

**Contradiction 1: Number of Casual Leave Days**

*   **Explanation:** Document A, the general Employee Handbook, states a different number of casual leave days than Document B, the specific Leave Policy. Document B explicitly states that it "supersedes earlier departmental guidance," implying it is the more current and authoritative policy regarding leave entitlements.
*   **Evidence from Document A:**
    *   Under "Leave": "Employees are eligible for **10 casual leave days**, 18 annual leave days and 10 sick leave days per calendar year."
*   **Evidence from Document B:**
    *   Under "Casual Leave": "Employees receive **12 casual leave days** each year. This policy supersedes earlier departmental guidance."


In [ ]:
# 13. Multilingual Demonstration

In [59]:
response = ask_question(
    "कितनी कैजुअल छुट्टियां मिलती हैं?"
)

response

{'question': 'कितनी कैजुअल छुट्टियां मिलती हैं?',
 'answer': 'दस्तावेज़ों में कैजुअल छुट्टियों की संख्या को लेकर विरोधाभास है।\n\n*   Leave_Policy-2.pdf दस्तावेज़ के अनुसार, कर्मचारियों को प्रति वर्ष 12 कैजुअल छुट्टियां मिलती हैं।\n*   Employee_Handbook-2.pdf दस्तावेज़ के अनुसार, कर्मचारी प्रति कैलेंडर वर्ष 10 कैजुअल छुट्टियों के लिए पात्र हैं।',
 'confidence': 0.43,
 'citations': [{'source': 'Leave_Policy-2.pdf',
   'page': 1,
   'snippet': 'Employees receive 12 casual leave days each year. This policy supersedes earlier departmental\nguidance. This policy applies to all full-time employees unless an exception is approved in writing\nby Hum...'},
  {'source': 'Employee_Handbook-2.pdf',
   'page': 1,
   'snippet': 'on the circumstances. These guidelines are reviewed periodically to ensure operational consistency\nand regulatory compliance.\nLeave\nEmployees are eligible for 10 casual leave days, 18 annual leave day...'}]}

In [60]:
response = ask_question(
    "किती कॅज्युअल लीव्ह मिळतात?"
)

response

{'question': 'किती कॅज्युअल लीव्ह मिळतात?',
 'answer': 'दिलेल्या कागदपत्रांमध्ये विसंगती आहे:\n\n*   **Leave_Policy-2.pdf** (पान 1) नुसार, कर्मचाऱ्यांना दरवर्षी 12 कॅज्युअल लीव्हचे दिवस मिळतात.\n*   **Employee_Handbook-2.pdf** (पान 1) नुसार, कर्मचारी प्रति कॅलेंडर वर्षासाठी 10 कॅज्युअल लीव्ह दिवसांसाठी पात्र आहेत.',
 'confidence': 0.43,
 'citations': [{'source': 'Leave_Policy-2.pdf',
   'page': 1,
   'snippet': 'Employees receive 12 casual leave days each year. This policy supersedes earlier departmental\nguidance. This policy applies to all full-time employees unless an exception is approved in writing\nby Hum...'},
  {'source': 'Employee_Handbook-2.pdf',
   'page': 1,
   'snippet': 'on the circumstances. These guidelines are reviewed periodically to ensure operational consistency\nand regulatory compliance.\nLeave\nEmployees are eligible for 10 casual leave days, 18 annual leave day...'}]}

In [61]:
response = ask_question(
    "எத்தனை சாதாரண விடுப்புகள் கிடைக்கும்?"
)

response

{'question': 'எத்தனை சாதாரண விடுப்புகள் கிடைக்கும்?',
 'answer': 'ஆவணங்கள் சாதாரண விடுப்புகளின் எண்ணிக்கையில் முரண்படுகின்றன.\n\n*   **Leave_Policy-2.pdf** (பக்கம் 1) கூறுகிறது, ஊழியர்கள் ஒவ்வொரு ஆண்டும் 12 சாதாரண விடுப்பு நாட்களைப் பெறுகிறார்கள்.\n*   **Employee_Handbook-2.pdf** (பக்கம் 1) கூறுகிறது, ஊழியர்கள் ஒரு காலண்டர் ஆண்டுக்கு 10 சாதாரண விடுப்பு நாட்களுக்குத் தகுதியுடையவர்கள்.',
 'confidence': 0.22,
 'citations': [{'source': 'Leave_Policy-2.pdf',
   'page': 1,
   'snippet': 'Employees receive 12 casual leave days each year. This policy supersedes earlier departmental\nguidance. This policy applies to all full-time employees unless an exception is approved in writing\nby Hum...'},
  {'source': 'Employee_Handbook-2.pdf',
   'page': 1,
   'snippet': 'on the circumstances. These guidelines are reviewed periodically to ensure operational consistency\nand regulatory compliance.\nLeave\nEmployees are eligible for 10 casual leave days, 18 annual leave day...'}]}

In [ ]:
# 14. Conclusion

This notebook demonstrates:

- Retrieval-Augmented Generation
- Semantic Search
- Citation-based Question Answering
- Document Contradiction Detection
- Multilingual Query Support
- Confidence Score